In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import glob
import numpy as np #1.23.5
import pandas as pd
import math
import matplotlib.pyplot as plt
import celldancer as cd
import celldancer.cdplt as cdplt
from celldancer.cdplt import colormap
import scanpy as sc
import scvelo as scv
import anndata
import os.path
import pickle as pickle
import celldancer.utilities as cdutil
import time
from os.path import exists
import unitvelo as utv
import dynamo as dyn 
method = 'cellDancer'
import warnings
warnings.filterwarnings('ignore')
from typing_extensions import Literal
from celldancer.utilities import export_velocity_to_dynamo

In [ ]:
#dataset name
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
#data path
data_dir = '/data/'
#result path
save_dir = '/result/'

In [ ]:

df_CB= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_CB)
df_IC= pd.DataFrame(columns=['Mean', 'Time(s)'])
print(df_IC)

Empty DataFrame
Columns: [Mean, Time(s)]
Index: []
Empty DataFrame
Columns: [Mean, Time(s)]
Index: []


In [ ]:
    for dataset in datasets:
        print(dataset)
        adata = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
        start = time.time()
        scv.pp.filter_genes(adata, min_shared_counts=20)
        scv.pp.filter_genes_dispersion(adata, n_top_genes=2000,log=False)
        dyn.tl.neighbors(adata,n_neighbors=30)
        scv.pp.moments(adata)
        cdutil.adata_to_df_with_embed(adata,
                                us_para=['Mu','Ms'],
                                cell_type_para='clusters',
                                embed_para='X_umap',
                                save_path=f'{save_dir}cellDancer/data/{dataset}_cell_type_u_s.csv')
        cell_type_u_s=pd.read_csv(f'{save_dir}cellDancer/data/{dataset}_cell_type_u_s.csv')
        os.makedirs(f'{save_dir}cellDancer/data/{dataset}_cellDancer_estimation', exist_ok=True)
        loss_df, cellDancer_df=cd.velocity(cell_type_u_s,
                                    permutation_ratio=0.125,
                                    n_jobs=8,
                                    save_path=f'{save_dir}cellDancer/data/{dataset}_cellDancer_estimation')
        cellDancer_df=pd.read_csv(f'{save_dir}cellDancer/data/{dataset}_cellDancer_estimation/cellDancer_estimation.csv')
        cellDancer_df=cd.compute_cell_velocity(cellDancer_df=cellDancer_df, projection_neighbor_size=100)
        end = time.time()
        adata_dyn = sc.read_h5ad(data_dir + dataset +'/'+ f'{dataset}.h5ad')
        dyn.pp.recipe_monocle(adata_dyn, n_top_genes=1000, fg_kwargs={'shared_count': 20})
        dyn.tl.dynamics(adata_dyn, model='stochastic')
        dyn.tl.neighbors(adata_dyn,n_neighbors=30)
        adata = export_velocity_to_dynamo(cellDancer_df,adata_dyn)
        print('                                             ')
        dyn.tl.reduceDimension(adata, n_pca_components=30)
        dyn.tl.cell_velocities(adata, basis="pca")
        
        fix, ax = plt.subplots(1, 1, figsize = (8, 6))
        dyn.pl.streamline_plot(adata, color=['clusters'], basis='umap', show_legend='on data', show_arrowed_spines=True,ax=ax)
        plt.savefig(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_{method}.svg')
        # Calculate performance metrics:
        adata.obsm['velocity_S_umap']=adata.obsm['velocity_umap']
        file = open(data_dir + dataset +'/'+f'{dataset}_groundTruth.pickle' ,'rb')
        ground_truth = pickle.load(file)
        metrics = utv.evaluate(adata, ground_truth, 'clusters', 'velocity_S')
        if exists(save_dir + method +'/'+ '_CBDir_scores.csv'):
            tab = pd.read_csv(save_dir + method +'/'+'_CBDir_scores.csv', index_col = 0)
        else:
            tab_cb = pd.DataFrame(columns = list(metrics['Cross-Boundary Direction Correctness (A->B)'].keys())  + ['Mean', 'Time(s)'],
                    index = [dataset])
            tab_IC = pd.DataFrame(columns = list(metrics['In-cluster Coherence'].keys())  + ['Mean', 'Time(s)'],
                    index = [dataset])
        ##CBDC_scores
        cb_score = [np.mean(metrics['Cross-Boundary Direction Correctness (A->B)'][x])
                    for x in metrics['Cross-Boundary Direction Correctness (A->B)'].keys()]
        tab_cb.loc[dataset,:] = cb_score + [np.mean(cb_score), end-start]
        #将新的数据行添加到Dataframe中
        df_CB = pd.concat([df_CB, pd.DataFrame([[np.mean(cb_score), end - start]], columns=df_CB.columns, index=[dataset])])
        tab_cb.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_CBDir_scores.csv')
        ##ICCoh_scores
        IC_score = [np.mean(metrics['In-cluster Coherence'][x])
                    for x in metrics['In-cluster Coherence'].keys()]
        tab_IC.loc[dataset,:] = IC_score + [np.mean(IC_score), end-start]
        #将新的数据行添加到Dataframe中
        df_IC = pd.concat([df_IC, pd.DataFrame([[np.mean(IC_score), end - start]], columns=df_IC.columns, index=[dataset])])
        tab_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{dataset}_ICCoh_scores.csv')
        
        adata.write_h5ad(save_dir + method +'/' +'CB_IC/'+ f'{dataset}_AnnData_Forscore.h5ad')

In [ ]:
df_CB.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_CBDir_scores.csv')
df_IC.to_csv(save_dir + method +'/'+'CB_IC/'+ f'{method}_ICCoh_scores.csv')